# Lab 4 essentials: an answer service that proves where its answer came from

**Time: about 50 minutes. Click Run all, then read each step while it runs. Do Lab 3 essentials first.**

**What you will do.** Build the answer service FreshCart's support agents need: it takes a question,
finds the best pieces with Lab 3's search, has a real language model write the answer with a source
identifier after each fact, and then **checks in code** that every identifier was really retrieved.
You break it on purpose with a fake source, measure it on 25 test questions, and find the answers the
code reads wrongly. Along the way you will find two problems that no error message tells you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| Lab 3's libraries | the search: meaning plus keyword search, the reranker, the recency filter | 1 | see Lab 3 essentials |
| `databricks-sdk` (already on Databricks) | `WorkspaceClient`: calls the language models this workspace serves, signed in as you, no key | 3 | [docs.databricks.com](https://docs.databricks.com/aws/en/dev-tools/sdk-python) |
| `langchain-core` | `ChatPromptTemplate` (the instructions with blanks), `RunnableLambda` (turns any Python function into a LangChain step), and the `|` that chains steps together | 2, 3 | [python.langchain.com](https://python.langchain.com/docs/concepts/lcel/) |
| `re` (built into Python) | finds every `[identifier]` in an answer | 4 | [docs.python.org](https://docs.python.org/3/library/re.html) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell.

**Five words used throughout.**

- **Grounding**: the model may only use the pieces it was given, not its memory.
- A **citation** is the identifier of the exact piece a fact came from, written in square brackets.
- The **citation check** is code that compares every cited identifier with the identifiers that were
  really retrieved. A model can write a made-up identifier that looks exactly like a real one.
- A **refusal** is the service saying it will not answer, with a reason. For a question the documents
  cannot answer, refusing is the right result.
- The **match score** is the reranker's score for the best piece. A low one means nothing relevant was
  found, so the service can refuse before paying for a model call.

**Cost.** The model is served by Databricks and billed per use: this notebook makes about 30 calls,
a few cents in total. Every answer is remembered, so later steps reuse it instead of paying again.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above, then restarts Python. It takes one
to two minutes. The restart is expected.

In [ ]:
# Lab 3's libraries: loaders, splitter, embedding model, keyword search, reranker.
# The model call needs nothing extra: databricks-sdk is already installed on Databricks.
%pip install -q langchain-core langchain-community langchain-classic langchain-text-splitters pypdf docx2txt python-pptx openpyxl fastembed rank_bm25 flashrank

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
dbutils.library.restartPython()

In [ ]:
# os lets us set settings for this session; warnings controls which notices are shown.
import os, warnings

# Hide notices that are not errors, such as messages about packages being renamed.
warnings.filterwarnings("ignore")

# Turn off the download progress bars of the embedding and reranker models.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
print("ready")

## Step 1. The search from Lab 3, giving 5 pieces with identifiers

**What this does.** Rebuilds Lab 3's best search in one cell: meaning plus keyword search, merged by
rank, then the reranker, over current documents only. It keeps the **top 5** pieces. Each piece also
gets an **identifier**: the first 12 characters of a fingerprint of its text, so the same piece always
gets the same identifier.

**What you should see.** `18 current pieces` (the 2022 policy's pieces are left out), then 5 pieces
for the refund question. The two from the 2024 refund policy have match scores near 1.0 and 0.8; the
other three are below 0.02. The reranker is confident about two pieces and not about the rest.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# The same imports as Lab 3 essentials, plus hashlib for the fingerprints.
import zipfile, pathlib, hashlib
import pandas as pd
from pptx import Presentation
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever

# Copy the seven documents and read all four formats, as in Lab 1.
PACK = PACK_ZIP
DOCS = pathlib.Path("/tmp/lab04_essentials/docs")
DOCS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as z:
    for name in z.namelist():
        if name.startswith("data/raw/") and not name.endswith("/"):
            (DOCS / pathlib.Path(name).name).write_bytes(z.read(name))
docs = []
for f in sorted(DOCS.glob("*.pdf")):
    docs += PyPDFLoader(str(f)).load()
docs += Docx2txtLoader(str(DOCS / "support_sla_matrix.docx")).load()
for slide in Presentation(DOCS / "support_qbr_q1_2024.pptx").slides:
    text = "\n".join(shape.text_frame.text for shape in slide.shapes if shape.has_text_frame)
    docs.append(Document(page_content=text, metadata={"source": "support_qbr_q1_2024.pptx"}))
table = pd.read_excel(DOCS / "refund_claims_march_2024.xlsx")
docs.append(Document(page_content=table.to_string(index=False), metadata={"source": "refund_claims_march_2024.xlsx"}))

# Cut into pieces, keep only current documents, and give each piece its labels and identifier.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pieces = []
for p in splitter.split_documents(docs):
    file = pathlib.Path(p.metadata["source"]).name
    # Skip empty pieces, and the replaced 2022 policy (Lab 3's recency filter).
    if not p.page_content.strip() or file == "refund_policy_v2_2022.pdf":
        continue
    # sha256 turns the text into a fingerprint; the first 12 characters are the identifier.
    piece_id = hashlib.sha256(p.page_content.encode()).hexdigest()[:12]
    p.metadata = {"file": file, "id": piece_id}
    pieces.append(p)

# Lab 3's search: meaning plus keyword, merged by rank, then the reranker keeps the best 5.
embedder = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
meaning = InMemoryVectorStore.from_documents(pieces, embedder).as_retriever(search_kwargs={"k": 10})
keyword = BM25Retriever.from_documents(pieces, k=10)
hybrid = EnsembleRetriever(retrievers=[keyword, meaning], weights=[0.5, 0.5])
reranker = FlashrankRerank(top_n=5, model="ms-marco-MiniLM-L-12-v2")
search = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=hybrid)

# Try it on one question. The reranker adds its score to each piece as "relevance_score".
question = "How many refund claims can a customer make in thirty days?"
found = search.invoke(question)
print(len(pieces), "current pieces;", len(found), "found for:", question)
for p in found:
    print(f"  match score {p.metadata['relevance_score']:.3f}   [{p.metadata['id']}]   {p.metadata['file']}")

## Step 2. The instructions for the model: a prompt with blanks

**What this does.** `ChatPromptTemplate` holds the instructions with two blanks, `{context}` and
`{question}`. The instructions are the **grounding rules**: use only the pieces given, cite the
identifier after every fact, and say so when the answer is not there. Each piece is written into the
context with its identifier in front, so the model can copy it.

**What you should see.** The full prompt exactly as the model will receive it, with the 5 pieces from
Step 1 filled in.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# The grounding rules, as the "system" message; the pieces and the question as the "human" message.
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context. "
               "After every fact, cite the identifier of the piece it came from, in square brackets, "
               "one identifier per bracket, for example [3f2a9c1e5b7d]. "
               "If the context does not contain the answer, reply exactly: I could not find the answer."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

# Write the pieces into the context, each starting with its identifier in brackets.
def as_context(found_pieces):
    return "\n\n".join(f"[{p.metadata['id']}] {p.page_content}" for p in found_pieces)

# Fill the blanks for our question and print what the model will see.
filled = prompt.invoke({"context": as_context(found), "question": question})
for message in filled.to_messages():
    print(f"--- {message.type} ---\n{message.content[:900]}\n")

## Step 3. The model writes the answer

**What this does.** Sends the filled prompt to `databricks-meta-llama-3-3-70b-instruct`, a language
model this workspace serves. `WorkspaceClient` signs in as you, so there is no key. `RunnableLambda`
turns that call into a LangChain step, and `prompt | model` chains the steps: fill the prompt, then
send it to the model. The function remembers every answer, so asking the same thing again costs nothing.

LangChain's own Databricks connector, `databricks-langchain`, is the usual way to do this, but on this
Databricks environment it cannot be installed (tested 13 October 2026). A `RunnableLambda` around one
Python function does the same job, and shows exactly what the connector does inside.

**What you should see.** One sentence, close to: "A customer may make up to three refund claims in
any rolling thirty day period [7c761f4760d9]." The identifier is copied from the context, exactly as
the prompt asked.

In [ ]:
# time.sleep pauses before a retry; WorkspaceClient talks to this Databricks workspace as you.
import time
from databricks.sdk import WorkspaceClient
from langchain_core.runnables import RunnableLambda

workspace = WorkspaceClient()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
# Every answer, kept by its exact messages, so the same question is never paid for twice.
answers_seen = {}

def call_model(filled_prompt):
    # The filled prompt as a list of {"role", "content"} messages, the format the served model expects.
    roles = {"system": "system", "human": "user", "ai": "assistant"}
    messages = [{"role": roles[m.type], "content": m.content} for m in filled_prompt.to_messages()]
    key = str(messages)
    if key in answers_seen:
        return answers_seen[key]
    # The served models allow only a few calls a second for the whole workspace: wait and retry if refused.
    for attempt in range(5):
        try:
            reply = workspace.api_client.do("POST", f"/serving-endpoints/{MODEL}/invocations",
                                            body={"messages": messages, "temperature": 0, "max_tokens": 400})
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 * (attempt + 1))
    # temperature=0 asks for the most likely wording, so answers vary less between runs.
    answers_seen[key] = reply["choices"][0]["message"]["content"]
    return answers_seen[key]

# The model as a LangChain step, and the chain: fill the prompt, then call the model.
model = RunnableLambda(call_model)
writer = prompt | model

answer = writer.invoke({"context": as_context(found), "question": question})
print(answer)

## Step 4. Check every citation in code, and return six fields

**What this does.** Puts the whole service in one function, `ask`. Two things can make it refuse:

1. **The gate:** the best match score is below a threshold (0.1 here), so nothing relevant was found
   and the model is not called at all.
2. **The citation check:** after the model answers, `re.findall` pulls out every `[identifier]`. Any
   identifier that was not among the 5 retrieved means the answer is refused, and so does an answer
   with no identifier at all.

Every answer comes back as the same six fields, refused or not.

**What you should see.** The six fields. `refused` is false, `citations` holds one identifier, and
that identifier is the first one in `retrieved`. `confidence` is the best match score, 1.0 here.

In [ ]:
# re finds patterns in text; json prints the six fields neatly.
import re, json

# A citation: "[" + 12 letters or digits + "]".
CITATION = re.compile(r"\[([0-9a-f]{12})\]")

def ask(question, threshold=0.1, writer=writer):
    found_pieces = search.invoke(question)
    retrieved = [p.metadata["id"] for p in found_pieces]
    # float() turns the reranker's numpy number into a plain Python number.
    best = float(found_pieces[0].metadata["relevance_score"]) if found_pieces else 0.0
    # The six fields, filled in as we go.
    result = {"answer": None, "citations": [], "confidence": round(best, 3),
              "retrieved": retrieved, "refused": True, "refusal_reason": None}
    # Refusal 1, the gate: nothing relevant enough, so do not pay for a model call.
    if best < threshold:
        result["refusal_reason"] = "no sufficiently relevant context"
        return result
    text = writer.invoke({"context": as_context(found_pieces), "question": question})
    cited = CITATION.findall(text)
    # Refusal 2, the citation check: an identifier that was never retrieved, or none at all.
    made_up = [c for c in cited if c not in retrieved]
    if made_up:
        result["refusal_reason"] = f"unverifiable citations: {made_up}"
    elif not cited:
        result["refusal_reason"] = "no citation produced"
    else:
        result.update(answer=text, citations=cited, refused=False)
    # Keep the model's own words too, for the checks in Step 7.
    result["model_wrote"] = text
    return result

print(json.dumps({k: v for k, v in ask(question).items() if k != "model_wrote"}, indent=1))

## Step 5. Break it on purpose: a made-up source

**What this does.** Wraps the model so that every third answer gets an extra identifier,
`[deadbeef0000]`, which was never retrieved. It looks exactly like a real one. Then it asks 9
questions through the service and counts how many fake citations the check catches.

**What you should see.** `fake citations added: 3, caught by the citation check: 3`. Every answer
with a made-up source was refused. Nothing raised an error, and the text of those answers looked
completely normal: only the check in code could tell.

In [ ]:
from langchain_core.runnables import RunnableLambda

# A counter shared across calls, and a step that adds a fake citation to every third answer.
counter = {"n": 0}
def add_fake_citation(text):
    counter["n"] += 1
    return text + " [deadbeef0000]" if counter["n"] % 3 == 0 else text

# The same chain as Step 3, with the fake-citation step at the end.
broken_writer = writer | RunnableLambda(add_fake_citation)

# Nine real questions, sent through the service with the broken writer.
nine = ["How many refund claims can a customer make in thirty days?",
        "When is wallet credit applied?",
        "What is the status of claim CLM-2024-0314?",
        "Within how many hours must photographic evidence of spillage be submitted?",
        "How long does a partner have to dispute a cost allocation?",
        "What was the refund rate in Q1 2024 according to the QBR?",
        "How is the cost shared when responsibility for a failure cannot be determined?",
        "What does spillage mean under the current refund policy?",
        "How many working days does it take for an approved refund to be credited?"]
# Ask each question once and count the refusals whose reason names the fake identifier.
caught = 0
for q in nine:
    r = ask(q, writer=broken_writer)
    if r["refusal_reason"] and "deadbeef0000" in r["refusal_reason"]:
        caught += 1
print(f"fake citations added: {counter['n'] // 3}, caught by the citation check: {caught}")

## Step 6. Measure it on 25 questions

**What this does.** Asks the honest service the 25 test questions: 20 the documents can answer, each
with the phrases a right answer must contain, and 5 they cannot. It reports three numbers, kept
separate on purpose:

- **Refused, answerable** (want low): good questions the service would not answer.
- **Refused, unanswerable** (want high): questions it rightly would not answer.
- **Right fact** (want high): answers that contain one of the expected phrases.

**What you should see.** Close to: 2 or 3 of 20 answerable questions refused, all 5 unanswerable
questions refused, and 17 or 18 of 20 right facts. The numbers can move by one between runs, even with
temperature 0. The full Lab 4, with a looser prompt, refused 1 of 20 and 4 of 5: this service is
stricter, which Step 7 explains.

In [ ]:
# Read the 25 test questions out of the course zip file.
with zipfile.ZipFile(PACK) as z:
    tests = json.loads(z.read("data/golden/qa_golden.json"))

rows = []
for group in ("answerable", "unanswerable"):
    for item in tests[group]:
        r = ask(item["question"])
        text = (r["answer"] or "").lower()
        # Right fact: the answer contains one of the expected phrases (answerable questions only).
        right = any(e.lower() in text for e in item.get("expect_any", [])) if group == "answerable" else None
        rows.append({"id": item["id"], "group": group, "refused": r["refused"],
                     "confidence": r["confidence"], "right fact": right, "result": r})

answerable = [r for r in rows if r["group"] == "answerable"]
unanswerable = [r for r in rows if r["group"] == "unanswerable"]
print(f"refused, answerable   : {sum(r['refused'] for r in answerable)}/20   (want low)")
print(f"refused, unanswerable : {sum(r['refused'] for r in unanswerable)}/5    (want high)")
print(f"right fact            : {sum(bool(r['right fact']) for r in answerable)}/20  (want high)")

### Step 7. Check: every refusal, and whether the code read the model right

**What this does.** Lists every refusal with its reason and the model's own words, so you can judge
each one. Then it looks for the two ways code misreads a good model: a refused answer that contains the
right fact, and an accepted answer whose words say the model could not find it.

**What you should see.**

- **Four unanswerable questions** refused because the model wrote "I could not find the answer" with
  no citation, and one, U03 (the chief executive's email), refused at the gate before any model call,
  with a match score of 0.017. Note U01, the Singapore question: its match score is 0.994, yet the
  documents cannot answer it. The score alone could not have refused it (Lab 3, Step 7b again).
- **Two or three answerable questions** refused although **the right fact was in the pieces the model
  was given**: A10 (the refund rate, 4.2) and A18 (the volume rise, 18), sometimes A02 too. The model
  was over-cautious.
- **0 disagreements.** The prompt's rules, "one identifier per bracket" and "reply exactly: I could not
  find the answer", stopped the two misreads the full Lab 4 shows (an answer with two identifiers in one
  bracket, and a "could not find" that still cited a piece).

**Try this.** In Step 2, delete `reply exactly: ` from the prompt and run Steps 2 to 7 again. Count
the refusals and the disagreements. Prompt rules trade one failure for another, which is why both
refusal rates are reported separately.

In [ ]:
# Phrases that mean the model is saying "no".
said_no = ("could not find", "cannot find", "not mentioned", "does not mention", "no information")

print("EVERY REFUSAL:")
for r in rows:
    if r["refused"]:
        wrote = r["result"].get("model_wrote", "(model not called)")
        print(f"  {r['id']} ({r['group']}), match score {r['confidence']:.3f}: {r['result']['refusal_reason']}")
        print(f"      model wrote: {wrote[:160]}")
        # For a question that has an answer: was the expected phrase in the pieces the model was given?
        if r["group"] == "answerable":
            item = next(i for i in tests["answerable"] if i["id"] == r["id"])
            given = as_context(search.invoke(item["question"])).lower()
            in_context = [e for e in item["expect_any"] if e.lower() in given]
            print(f"      the right fact was in the pieces it was given: {bool(in_context)} {in_context}")

print("\nWHERE THE CODE AND THE MODEL DISAGREE:")
disagreements = 0
for r in rows:
    wrote = r["result"].get("model_wrote", "")
    item = next(i for i in tests[r["group"]] if i["id"] == r["id"])
    has_fact = any(e.lower() in wrote.lower() for e in item.get("expect_any", []))
    # A right answer that the service refused anyway.
    if r["refused"] and r["group"] == "answerable" and has_fact:
        disagreements += 1
        print(f"  {r['id']} refused, but the model's answer had the right fact: {wrote[:160]}")
    # An answer the service accepted although the model said it could not find it.
    if not r["refused"] and any(p in wrote.lower() for p in said_no):
        disagreements += 1
        print(f"  {r['id']} accepted, but the model said it could not find it: {wrote[:160]}")
print(f"  {disagreements} found")

## What you found

1. **Citations checked in code catch made-up sources** (Step 5): 3 of 3, while the answers looked
   normal to a reader.
2. **The model's "I could not find the answer" did most of the refusing** (Step 7): 4 of 5. The score
   gate caught only the one question with nothing relevant at all; the Singapore question scored 0.994.
3. **A stricter prompt removes misreads but adds over-refusals** (Step 7): 2 or 3 good questions were
   refused although the answer was in front of the model.
4. **Results move between runs** (Step 6), even at temperature 0. Run a test set more than once before
   trusting a difference of one question.

**Where this goes next.** The full Lab 4 notebook, `lab04_rag_api`, measures the same service with
the course's own code, adds supported sentences and citation precision and recall, and runs it as an
API on your own computer (see **Lab 4 as an API** in the README).

## Look it up yourself: every setting the libraries offer

**What this does.** For the main things this lab calls, the cell prints every parameter (read from the
installed library), the values this lab passed, and one setting worth trying.

**What you should see.** Four short blocks.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap

# parameters(thing): the parameters `thing` accepts, as text such as "temperature=None".
def parameters(thing):
    # Some classes list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        return [n if f.is_required() else f"{n}={f.get_default(call_default_factory=True)!r}"
                for n, f in thing.model_fields.items()]
    try:
        sig = inspect.signature(thing)
    except (TypeError, ValueError):
        return []
    # Skip "self", "cls", and the *args / **kwargs catch-alls, which are not named settings.
    return [p.name if p.default is p.empty else f"{p.name}={p.default!r}" for p in sig.parameters.values()
            if p.name not in ("self", "cls") and p.kind not in (p.VAR_POSITIONAL, p.VAR_KEYWORD)]

# look_up prints the three lines for one function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    print(name)
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

look_up("RunnableLambda", RunnableLambda, "func=call_model",
        "any Python function can become a step and be chained with |")
look_up("ChatPromptTemplate.from_messages", ChatPromptTemplate.from_messages, '[("system", rules), ("human", context and question)]',
        "the messages list: add an example question and answer to show the model the citation format")
look_up("FlashrankRerank", FlashrankRerank, 'top_n=5, model="ms-marco-MiniLM-L-12-v2"',
        "top_n: how many pieces the model gets; more context costs more and can distract it")
look_up("re.findall", re.findall, "the CITATION pattern, and the answer text",
        "the pattern: make it read every identifier inside a bracket, the fix for Step 7")

## Your notes

Double click to type: which of the problems in Step 7 you would fix first, and how you would test the fix.